In [121]:
# import libraries
import warnings
from pathlib import Path
import subprocess

import nglview as nv
from openbabel import pybel

from opencadd.structure.core import Structure

In [122]:
import nglview as nv

# Load the local PDB file directly
view = nv.show_file("2ito.pdb")
view

NGLWidget()

In [123]:
import openbabel
print(openbabel.__version__)

3.1.0


In [124]:
from Bio.PDB import PDBParser, PDBIO, Select

class ProteinSelect(Select):
    """Custom filter to keep only protein atoms and exclude water/ligand."""
    def accept_residue(self, residue):
        # Keep standard amino acids, drop water (HOH) and heteroatoms
        resname = residue.get_resname().strip()
        return resname not in ["HOH", "LIG", "UNL"] # adjust LIG to match your ligand residue name

# 1. Parse the local file
parser = PDBParser(QUIET=True)
structure = parser.get_structure("2ITO", "2ito.pdb")

# 2. Save just the protein to a new file
io = PDBIO()
io.set_structure(structure)
io.save("protein_only.pdb", ProteinSelect())

print("Protein successfully saved as protein_only.pdb!")

Protein successfully saved as protein_only.pdb!


In [125]:
from Bio.PDB import PDBParser, PDBIO, Select
from rdkit import Chem
import numpy as np

# 1. Create a filter to grab ONLY the ligand (anything that isn't water or a standard amino acid)
class LigandSelect(Select):
    def accept_residue(self, residue):
        resname = residue.get_resname().strip()
        standard_amino_acids = ["ALA", "ARG", "ASN", "ASP", "CYS", "GLU", "GLN", "GLY", "HIS", "ILE", "LEU", "LYS", "MET", "PHE", "PRO", "SER", "THR", "TRP", "TYR", "VAL"]
        return resname not in standard_amino_acids and resname != "HOH"

# 2. Extract the ligand from 2ito.pdb and save it as extracted_ligand.pdb
parser = PDBParser(QUIET=True)
structure = parser.get_structure("2ITO", "2ito.pdb")
io = PDBIO()
io.set_structure(structure)
io.save("extracted_ligand.pdb", LigandSelect())

In [150]:
def pdb_to_pdbqt(pdb_path, pdbqt_path, pH=7.4):
    """
    Convert a PDB file to a PDBQT file needed by docking programs of the AutoDock family.

    Parameters
    ----------
    pdb_path: str or pathlib.Path
        Path to input PDB file.
    pdbqt_path: str or pathlib.path
        Path to output PDBQT file.
    pH: float
        Protonation at given pH.
    """
    molecule = list(pybel.readfile("pdb", str(pdb_path)))[0]
    # add hydrogens at given pH
    molecule.OBMol.CorrectForPH(pH)
    molecule.addh()
    # add partial charges to each atom
    for atom in molecule.atoms:
        atom.OBAtom.GetPartialCharge()
    molecule.write("pdbqt", str(pdbqt_path), overwrite=True)
    return

In [151]:
# convert extracted ligand to PDBQT format
pdb_to_pdbqt("extracted_ligand.pdb", "ligand.pdbqt")

In [152]:
def run_smina(
    ligand_path, protein_path, out_path, pocket_center, pocket_size, num_poses=10, exhaustiveness=8
):
    """
    Perform docking with Smina.

    Parameters
    ----------
    ligand_path: str or pathlib.Path
        Path to ligand PDBQT file that should be docked.
    protein_path: str or pathlib.Path
        Path to protein PDBQT file that should be docked to.
    out_path: str or pathlib.Path
        Path to which docking poses should be saved, SDF or PDB format.
    pocket_center: iterable of float or int
        Coordinates defining the center of the binding site.
    pocket_size: iterable of float or int
        Lengths of edges defining the binding site.
    num_poses: int
        Maximum number of poses to generate.
    exhaustiveness: int
        Accuracy of docking calculations.

    Returns
    -------
    output_text: str
        The output of the Smina calculation.
    """
    output_text = subprocess.check_output(
        [
            "smina",
            "--ligand",
            str(ligand_path),
            "--receptor",
            str(protein_path),
            "--out",
            str(out_path),
            "--center_x",
            str(pocket_center[0]),
            "--center_y",
            str(pocket_center[1]),
            "--center_z",
            str(pocket_center[2]),
            "--size_x",
            str(pocket_size[0]),
            "--size_y",
            str(pocket_size[1]),
            "--size_z",
            str(pocket_size[2]),
            "--num_modes",
            str(num_poses),
            "--exhaustiveness",
            str(exhaustiveness),
        ],
        universal_newlines=True,  # needed to capture output text
    )
    return output_text

In [153]:
import numpy as np
from rdkit import Chem

# 1. Load the crystal reference ligand from the active site
ref_mol = Chem.MolFromPDBFile("extracted_ligand.pdb", removeHs=False)
conf = ref_mol.GetConformer()
coords = np.array([conf.GetAtomPosition(i) for i in range(ref_mol.GetNumAtoms())])

# 2. Calculate the exact center of the pocket
center = np.mean(coords, axis=0)

# 3. Calculate the box size based on the ligand's dimensions + 6Å padding
min_coords = np.min(coords, axis=0)
max_coords = np.max(coords, axis=0)
box_size = (max_coords - min_coords) + 6.0

pocket_center = max_coords + min_coords / 2
pocket_size = max_coords - min_coords + 5

print(f"--pocket_center: {center[0]:.3f}")
print(f"--pocket_size: {box_size[0]:.3f}")

--pocket_center: -49.757
--pocket_size: 16.102


In [154]:
output_text = run_smina(
    "ligand.pdbqt",
    "protein_only.pdb",
    "docking_poses.sdf",
    pocket_center,
    pocket_size,
)
print(output_text)

   _______  _______ _________ _        _______ 
  (  ____ \(       )\__   __/( (    /|(  ___  )
  | (    \/| () () |   ) (   |  \  ( || (   ) |
  | (_____ | || || |   | |   |   \ | || (___) |
  (_____  )| |(_)| |   | |   | (\ \) ||  ___  |
        ) || |   | |   | |   | | \   || (   ) |
  /\____) || )   ( |___) (___| )  \  || )   ( |
  \_______)|/     \|\_______/|/    )_)|/     \|


smina is based off AutoDock Vina. Please cite appropriately.

Weights      Terms
-0.035579    gauss(o=0,_w=0.5,_c=8)
-0.005156    gauss(o=3,_w=2,_c=8)
0.840245     repulsion(o=0,_c=8)
-0.035069    hydrophobic(g=0.5,_b=1.5,_c=8)
-0.587439    non_dir_h_bond(g=-0.7,_b=0,_c=8)
1.923        num_tors_div

Using random seed: 898558025

0%   10   20   30   40   50   60   70   80   90   100%
|----|----|----|----|----|----|----|----|----|----|
***************************************************

mode |   affinity | dist from best mode
     | (kcal/mol) | rmsd l.b.| rmsd u.b.
-----+------------+----------+----------
1

In [155]:
def split_sdf_file(sdf_path):
    """
    Split an SDF file into seperate files for each molecule.
    Each file is named with consecutive numbers.

    Parameters
    ----------
    sdf_path: str or pathlib.Path
        Path to SDF file that should be split.
    """
    sdf_path = Path(sdf_path)
    stem = sdf_path.stem
    parent = sdf_path.parent
    molecules = pybel.readfile("sdf", str(sdf_path))
    for i, molecule in enumerate(molecules, 1):
        molecule.write("sdf", str(parent / f"{stem}_{i}.sdf"), overwrite=True)
    return

In [156]:
split_sdf_file("docking_poses.sdf")

In [164]:
import nglview as nv

# 1. Load the protein PDB file
view = nv.show_file("protein_only.pdb")
view.clear_representations()
view.add_representation("cartoon", selection="protein")

# 2. Add the docking poses SDF file using the correct method
docking_pose_id = 1
view.add_component(str(f"docking_poses_{docking_pose_id}.sdf"))


# 3. Style the ligand as licorice so you can see its atoms
view.add_representation("licorice", selection="not protein")

# Display the view
view

NGLWidget()